## Exercise 2: First look


In [1]:
# Cell 1: Imports

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
## Load Titanic dataset
df_raw = pd.read_csv("data/titanic.csv")
print(f"Shape of the dataset: {df_raw.shape}")
display(df_raw.head())

Shape of the dataset: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
# Cell 4: Column types and missing values
print("Column types and missing values:")
df_raw.info()

Column types and missing values:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


In [4]:
# Cell 5: Summary of the number columns
df_raw.describe().round(2) # .round(2) rounds the numbers to 2 decimal places

,survived,pclass,age,sibsp,parch,fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


In [ ]:
# Cell 6: Summary of the text columns

print(f"Summary of text columns:\n{df_raw.describe(include='str')}") # include='str' includes only text columns

Summary of text columns:
         sex embarked  class  who deck  embark_town alive
count    891      889    891  891  203          889   891
unique     2        3      3    3    7            3     2
top     male        S  Third  man    C  Southampton    no
freq     577      644    491  537   59          644   549


In [6]:
#  Cell 7: Percentage missing per column

missing_pct = (df_raw.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct[missing_pct > 0]

deck           77.2
age            19.9
embarked        0.2
embark_town     0.2
dtype: float64

In [7]:
#  Cell 8: Survival rate

survival_rate = df_raw["survived"].mean()
print(f"Survival rate: {survival_rate:.1%}")   
print(df_raw["survived"].value_counts())

Survival rate: 38.4%
survived
0    549
1    342
Name: count, dtype: int64


In [8]:
# Cell 9: Passengers per class and per port

print(f"Passengers per class:\n{df_raw['class'].value_counts()}")
print(f"Passengers per port:\n{df_raw['embarked'].value_counts(dropna=False)}") # dropna=False includes NaN values in the count

Passengers per class:
class
Third     491
First     216
Second    184
Name: count, dtype: int64
Passengers per port:
embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


## Exercise 2 answers
- **Size:** 891 passengers, 15 columns.
- **Missing:** `deck` 77.2%, `age` 19.9%, `embarked` / `embark_town` 0.2% (2 rows). Everything else is complete.
- **Survival rate:** 38.4% (342 of 891).
- **Class:** Third 491, First 216, Second 184. **Port:** Southampton 644, Cherbourg 168, Queenstown 77, unknown 2.
- **Fare is skewed:** mean £32.20 vs median £14.45 (max £512.33), so a few very expensive tickets pull the average up.
- `pclass` is a category stored as a number, so its mean (2.31) is meaningless.
- The mean of a True/False or 0/1 column is the share of `True`/1, which is how `isna().mean()` gives the share missing.


## Exercise 3: Spot the problems

In [9]:
# Part 1: Redundant columns
pd.crosstab(df_raw["pclass"], df_raw["class"])



class,First,Second,Third
pclass,,,
1,216,0,0
2,0,184,0
3,0,0,491


In [10]:
display(pd.crosstab(df_raw["survived"], df_raw["alive"]))
display(pd.crosstab(df_raw["embarked"], df_raw["embark_town"]))

alive,no,yes
survived,,
0,549,0
1,0,342


embark_town,Cherbourg,Queenstown,Southampton
embarked,,,
C,168,0,0
Q,0,77,0
S,0,0,644


In [11]:
# Cell 3: Derived columns

print((df_raw["adult_male"] == (df_raw["who"] == "man")).all())
print((df_raw["alone"] == (df_raw["sibsp"] + df_raw["parch"] == 0)).all())

True
True


## Part 2: Duplicates

In [12]:
# Cell 4 : How many?

print("Duplicated rows:", df_raw.duplicated().sum())
print("Rows involved:  ", df_raw.duplicated(keep=False).sum())

Duplicated rows: 107
Rows involved:   160


In [13]:
# Cell 5: The biggest groups

cols = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
dups = df_raw[df_raw.duplicated(keep=False)]
dups[cols].value_counts(dropna=False).head(3)

survived  pclass  sex   age  sibsp  parch  fare    embarked
0         3       male  NaN  0      0      7.8958  S           13
                                           8.0500  S           12
                                           7.7500  Q            8
Name: count, dtype: int64

In [14]:
# Cell 6: What the duplicates have in common

print(f"Age missing: {dups['age'].isna().mean():.1%}")
print(dups["pclass"].value_counts())

Age missing: 58.1%
pclass
3    125
2     31
1      4
Name: count, dtype: int64


## Part 3: Odd values

In [15]:
# Cell 7: Fares of £0

zero = df_raw[df_raw["fare"] == 0]
print(f"{len(zero)} passengers paid £0: "
      f"{(zero['sex'] == 'male').sum()} men, {zero['alone'].sum()} alone, "
      f"{(zero['embarked'] == 'S').sum()} from Southampton, {zero['survived'].sum()} survived")
print(zero["pclass"].value_counts().sort_index())

15 passengers paid £0: 15 men, 15 alone, 15 from Southampton, 1 survived
pclass
1    5
2    6
3    4
Name: count, dtype: int64


In [16]:
# Cell 8: The highest fares

display(df_raw.nlargest(5, "fare")[["survived", "pclass", "sex", "age", "fare", "embarked"]])
print("99% of fares are below:", df_raw["fare"].quantile(0.99).round(2))

,survived,pclass,sex,age,fare,embarked
258,1,1,female,35.0,512.3292,C
679,1,1,male,36.0,512.3292,C
737,1,1,male,35.0,512.3292,C
27,0,1,male,19.0,263.0000,S
88,1,1,female,23.0,263.0000,S


99% of fares are below: 249.01


In [17]:
# Bonus: Odd ages

print("Babies:", df_raw.loc[df_raw["age"] < 1, "age"].tolist())
print("Ages ending in .5 (over 1):", ((df_raw["age"] % 1 == 0.5) & (df_raw["age"] > 1)).sum())

Babies: [0.83, 0.92, 0.75, 0.75, 0.67, 0.42, 0.83]
Ages ending in .5 (over 1): 18


## Exercise 3 answers
- **Redundant pairs** (crosstabs are diagonal-only): `pclass`/`class`, `survived`/`alive`, `embarked`/`embark_town`. Keep one of each. `alive` would be **data leakage** if it were used to predict `survived`.
- **Derived columns:** `adult_male` = "who is man"; `alone` = sibsp + parch == 0. Keep them as shortcuts.
- **Duplicates:** 107 (160 rows in 53 groups; the biggest is 13 third-class men with no age and the same £7.90 fare). **Keep them:** with no name or ticket ID, identical-looking rows are different people. Deleting them would remove real passengers.
- **£0 fares:** 15 passengers, all men, alone, from Southampton (likely complimentary tickets). Real, not errors.
- **Max fare £512.33:** 3 first-class passengers from Cherbourg on one shared booking. A real outlier; 99% of fares are below £249.
- **Outlier ≠ error:** delete errors, keep real outliers, and use the median when data is skewed.
